# 08. Lecke - Többügynökös Tervezési Minta


## Beállítás


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Miért többügynökös rendszerek?

A valós feladatok, mint például az utazástervezés, sokféle szakértelmet igényelnek — logisztika, helyi ismeretek, költségvetés és még sok más. Egyetlen ügynök, aki mindent próbál kezelni, gyorsan kezelhetetlenné válik.

A többügynökös rendszerek ezt **szakosodás** révén oldják meg: minden ügynök egy adott területre fókuszál, így magasabb minőségű eredményeket produkál, mint egy általános szakember. Emellett javítják a **skálázhatóságot** is — új ügynököket adhatsz hozzá (például egy repülőjárat-szakértőt, étteremkritikust) anélkül, hogy át kellene írni a meglévő munkafolyamatot. Az ügynökök egy strukturált csővezetéken keresztül kapcsolódnak össze, átadva a kontextust egyikől a másikig.


## Speciális ügynökök létrehozása


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Szekvenciális munkafolyamat építése

A `WorkflowBuilder` lehetővé teszi az ügynökök irányított gráfba való bekötését. Itt egy egyszerű kétlépéses folyamatot hozunk létre: a **TravelPlanner** megtervezi az útitervet, majd a **TravelConcierge** felülvizsgálja és továbbfejleszti azt.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## További ügynökök hozzáadása a munkafolyamathoz

A multi-agent minta egyik legnagyobb előnye, hogy mennyire könnyen bővíthető. Alább hozzáadunk egy **BudgetReviewer** ügynököt, amely ellenőrzi a tervet az utazó költségvetése alapján, jelöli azokat a tételeket, amelyek esetleg meghaladják a költségkeretet, és pénztárcabarát alternatívákat javasol. A munkafolyamat most három ügynököt futtat egymás után:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Összefoglaló

Ebben a leckében megtanultad, hogyan:

1. **Hozz létre specializált ügynököket** — mindegyiknek egy fókuszált szereppel (tervezés, concierge, költségvetés felülvizsgálat).
2. **Fűzd össze az ügynököket egy egymást követő munkafolyamatba** a `WorkflowBuilder` és az `add_edge` használatával.
3. **Streameld a kimenetet** egy többügynökös csővezetékről, követve, hogy éppen melyik ügynök beszél.
4. **Bővítsd a munkafolyamatot** új ügynökök hozzáadásával a lánchoz anélkül, hogy a meglévőket módosítanád.

A többügynökös tervezési minta egyszerűen tartja az egyes ügynököket, miközben gazdagabb, alaposabban átvizsgált eredményeket produkál, mint amire egyetlen ügynök önmagában képes lenne.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Nyilatkozat**:  
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár a pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum anyanyelvű változata tekintendő hiteles forrásnak. Fontos információk esetén szakmai, emberi fordítást javasolunk. Nem vállalunk felelősséget a fordítás használatából eredő félreértésekért vagy félreértelmezésekért.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
